# RAG with LlamIndex - Exp1

* About: Make the whole RAG process runnable
* Using Ollama local LLMs, seems that my RAM is too limited to use larger models, but small models are very slow even just to generate the answer for one question

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ.pop("OPENAI_API_KEY", None)

from datasets import load_dataset
import pandas as pd
import json
from llama_index.core import (
    VectorStoreIndex,
    Settings,
)

from utils import *

import warnings
warnings.filterwarnings('ignore')

resource module not available on Windows


## Load FinanceBench Data

* [Huggingface FinanceBench data][1]
  * [data dictionary][2]
    * `justification` is the expected answer to be compared with `answer`
    * `evidence` is the evidence of justification
  * JSONL data
    * Text are processed markdown text
    * Has metadata ready
  * Download `pdfs/` [here][2]

[1]:https://huggingface.co/datasets/PatronusAI/financebench
[2]:https://github.com/patronus-ai/financebench/tree/main

In [2]:
# Load FinanceBench dataset from Hugging Face
dataset = load_dataset("PatronusAI/financebench")

# Explore the dataset structure
print(dataset)
print(dataset['train'].column_names)
for k, v in dataset['train'][0].items():
    print(f"{k}: {v}")
    if k == 'evidence':
        print(v[0].keys())
    print()

DatasetDict({
    train: Dataset({
        features: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link'],
        num_rows: 150
    })
})
['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']
financebench_id: financebench_id_03029

company: 3M

doc_name: 3M_2018_10K

question_type: metrics-generated

question_reasoning: Information extraction

domain_question_num: None

question: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

answer: $1577.00

justification: The metric capital expenditures was directly extracted from

In [3]:
# Choose the latest document from each company
selected_docs = list(get_latest_files_per_company(pdfs_folder='pdfs/').values())
print(len(selected_docs), selected_docs)

# Pre-load all selected PDFs once
loaded_pdf = {}
with ThreadPoolExecutor(max_workers=10) as executor:
    loaded_pdf = dict(executor.map(load_pdf_content, selected_docs))

print(len(loaded_pdf))

40 ['pdfs/ACTIVISIONBLIZZARD_2022_10K.pdf', 'pdfs/ADOBE_2022_10K.pdf', 'pdfs/AES_2022_10K.pdf', 'pdfs/AMAZON_2022_10K.pdf', 'pdfs/AMCOR_2023_10K.pdf', 'pdfs/AMD_2022_10K.pdf', 'pdfs/AMERICANEXPRESS_2022_10K.pdf', 'pdfs/AMERICANWATERWORKS_2022_10K.pdf', 'pdfs/APPLE_2022_10K.pdf', 'pdfs/BESTBUY_2023_10K.pdf', 'pdfs/BLOCK_2022_10K.pdf', 'pdfs/BOEING_2022_10K.pdf', 'pdfs/BOSTONPROPERTIES_2015_10K.pdf', 'pdfs/COCACOLA_2022_10K.pdf', 'pdfs/CORNING_2022_10K.pdf', 'pdfs/COSTCO_2023_8K_dated-2023-01-19.pdf', 'pdfs/CVSHEALTH_2022_10K.pdf', 'pdfs/EBAY_2022_10K.pdf', 'pdfs/FEDEX_2023_10K.pdf', 'pdfs/FOOTLOCKER_2023_10K.pdf', 'pdfs/GENERALMILLS_2023_10K.pdf', 'pdfs/INTEL_2023_8K_dated-2022-11-22.pdf', 'pdfs/JOHNSON_JOHNSON_2023_8K_dated-2023-08-23.pdf', 'pdfs/JPMORGAN_2022_10K.pdf', 'pdfs/KRAFTHEINZ_2022_10K.pdf', 'pdfs/LOCKHEEDMARTIN_2023_8K_dated-2023-05-25.pdf', 'pdfs/MCDONALDS_2022_10K.pdf', 'pdfs/MGMRESORTS_2023_8K_dated-2023-03-01.pdf', 'pdfs/MICROSOFT_2023_10K.pdf', 'pdfs/NETFLIX_2022_10K.pd

In [4]:
# Genrate LlamaIndex Documents from selected documents in the dataset
documents = []
selected_metadata_cols = ['company', 'doc_name', 'doc_period', 'doc_link']

documents = process_items_parallel(
    dataset['train'],
    selected_doc_names=set([selected_doc.replace('pdfs/', '').replace('.pdf', '')
                             for selected_doc in selected_docs]),
    loaded_pdf=loaded_pdf,
    selected_metadata_cols=selected_metadata_cols,
    max_workers=10
)

print(f"Generated {len(documents)} documents")

Generated 50 documents


## Run RAG

* Free components here
  * `llm`
  * `embed_model`
  * `reranker`
* Time Consuming Steps
  * vector indexing because of embedding
  * reranking

In [5]:
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import StorageContext, load_index_from_storage

In [6]:
llm = Ollama(model="gemma2:2b", request_timeout=120.0, temperature=0.1)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", device="cpu")
node_parser = setup_chunking_strategy(embed_model=embed_model)

Settings.llm = llm
Settings.embed_model = embed_model

print("Model name:", getattr(Settings.llm, "model", None))
print("Model name:", getattr(Settings.embed_model, "model_name", getattr(Settings.embed_model, "model_id", None)))

Model name: gemma2:2b
Model name: BAAI/bge-small-en-v1.5


In [7]:
# # Create vector store (40 mins ~ 1 hour to create)
# vector_index = VectorStoreIndex.from_documents(
#     documents,
#     node_parser=node_parser,
#     show_progress=True
# )

# # Save vector store indexing
# vector_index.storage_context.persist(persist_dir="./storage")
# print("Index saved to ./storage")

In [8]:
# Load vector store indexing
storage_context = StorageContext.from_defaults(persist_dir="./storage")
vector_index = load_index_from_storage(storage_context)

In [9]:
# Create hybrid retriever
retriever = HybridRetriever(
            vector_index,
            documents,
            top_k=5,
            alpha=0.6  # 60% weight to semantic retrieval (dense), 40% to key words retrieval (sparse)
        )


# reranker = CrossEncoderRerank(top_n=3)
# query_engine = get_query_engine(retriever, reranker)
# query_engine

query_engine = get_query_engine(retriever, reranker=None)  # without reranker
query_engine

In [ ]:
queries = [
        "What was Apple's revenue in 2023?",
        "Compare Microsoft's operating margin across Q1-Q4 2023",
        "What companies had the highest debt-to-equity ratio in 2023?",
    ]
    
for q in queries:
    response, retrieved_nodes = get_rag_response(query_engine, q)
    print(f"\nResponse:\n{response}")
    print(retrieved_nodes)
    break

In [ ]:
for retrieved_node in retrieved_nodes:
    print(retrieved_node.metadata)
    print(retrieved_node.get_content())
    print('--------------------------------------------------------------')
    break

## Evaluation

In [11]:
select_docs = set([selected_doc.replace('pdfs/', '').replace('.pdf', '') for selected_doc in selected_docs])
select_items = [item for item in dataset['train'] if item['doc_name'] 
               in select_docs]
print(len(select_items))

eval_lst = await run_eval_async(select_items, query_engine, concurrency=5)
print(len(eval_lst))

with open("output/exp1_raw.json", "w", encoding="utf-8") as f:
    json.dump(eval_lst, f, ensure_ascii=False, indent=2)

50
50
